# Preprocesamiento de Datos EMG - NinaPro Database

Este notebook carga y procesa los datos de señal EMG de las bases de datos NinaPro, 
creando dos datasets separados:
- **Intact**: Sujetos con manos intactas (DB1, DB2, DB4, DB5, DB7 sujetos 1-20)
- **Amputated**: Sujetos amputados (DB3, DB7 sujetos 21-22)

Se utilizan únicamente los canales 1-8 para mantener consistencia entre bases de datos.
2. Configuración:

## 1. Configuración e Importaciones

In [23]:
import pandas as pd
import numpy as np

## 2. Selección de Columnas

DB4 tiene 12 canales mientras las demás bases de datos tienen 8. 
Para mantener consistencia, solo se conservan los canales 1-8 (208 features).

In [ ]:
# Columnas de canales 1-8
keep_cols = [col for col in pd.read_csv('../data/processed/NinaProDB2.csv', nrows=0).columns 
             if not col.startswith('Channel 9_') and not col.startswith('Channel 10_') 
             and not col.startswith('Channel 11_') and not col.startswith('Channel 12_')]

## 3. Dataset Intact

| Database | Sujetos | Tipo |
|----------|---------|------|
| DB1 | 27 | 100% Intact |
| DB2 | 40 | 100% Intact |
| DB4 | 10 | 100% Intact |
| DB5 | 10 | 100% Intact |
| DB7 | 20 | Intact (sujetos 1-20) |

**Total: ~107 sujetos**

In [40]:
# Intact
intact_files = {'DB1': 'NInaProDB1.csv', 'DB2': 'NinaProDB2.csv', 
                'DB4': 'NinaProDB4.csv', 'DB5': 'NinaProDB5.csv'}


In [26]:
dfs_intact = []
for db, file in intact_files.items():
    df = pd.read_csv(f'../data/processed/{file}')[keep_cols]
    df['Database'] = db
    dfs_intact.append(df)

C:\Users\bernarda.salazar\AppData\Local\Temp\ipykernel_19856\240238965.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Database'] = db
C:\Users\bernarda.salazar\AppData\Local\Temp\ipykernel_19856\240238965.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Database'] = db
C:\Users\bernarda.salazar\AppData\Local\Temp\ipykernel_19856\240238965.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all col

In [27]:
# DB7 intact (subjects 1-20)
df_db7 = pd.read_csv('../data/processed/NinaProDB7.csv')[keep_cols]
df_db7['Database'] = 'DB7'
dfs_intact.append(df_db7[df_db7['subject'] <= 20])

C:\Users\bernarda.salazar\AppData\Local\Temp\ipykernel_19856\2937484689.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_db7['Database'] = 'DB7'


## 4. Dataset Amputados

| Database | Sujetos | Tipo |
|----------|---------|------|
| DB3 | 11 | 100% Amputados |
| DB7 | 2 | Amputados (sujetos 21-22) |

**Total: ~13 sujetos**

In [28]:
# Amputated
df_db3 = pd.read_csv('../data/processed/NinaProDB3.csv')[keep_cols]
df_db3['Database'] = 'DB3'
dfs_amp = [df_db3, df_db7[df_db7['subject'] >= 21]]

C:\Users\bernarda.salazar\AppData\Local\Temp\ipykernel_19856\1810923110.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_db3['Database'] = 'DB3'



## 5. Guardar Datasets
Los archivos se guardan en `data/master/`:
- `intact_signals.csv`
- `amputated_signals.csv`

In [29]:
# Save
pd.concat(dfs_intact).to_csv('../data/master/intact_signals.csv', index=False)
pd.concat(dfs_amp).to_csv('../data/master/amputated_signals.csv', index=False)

In [39]:
dataset = pd.DataFrame(dfs_intact)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (5,) + inhomogeneous part.

## 6. Metadata

In [30]:
# Columnas comunes para intact
INTACT_COLS = ['Subject','Hand', 'Handedness', 'Gender', 'Age', 'Height', 'Weight']

In [31]:
# ============ INTACT METADATA ============
intact_files = {
    'DB1': 'DB1SubjectData.csv',
    'DB2': 'DB2SubjectData.csv', 
    'DB4': 'DB4SubjectData.csv',
    'DB5': 'DB5SubjectData.csv'
}

In [32]:
dfs_intact = []
for db, file in intact_files.items():
    df = pd.read_csv(f'../data/metadata/{file}')
    df['Database'] = db
    dfs_intact.append(df)

In [33]:
# DB7 intact (subjects 1-20) - solo columnas comunes
df_db7 = pd.read_csv('../data/metadata/DB7SubjectData.csv')
df_db7['Database'] = 'DB7'
df_db7['Gender'] = df_db7['Height'].apply(lambda h: 'Male' if h >= 170 else 'Female')
dfs_intact.append(df_db7[df_db7['Subject'] <= 20][INTACT_COLS + ['Database']])

In [34]:
# ============ AMPUTATED METADATA ============
# DB3 (todas las columnas)
df_db3 = pd.read_csv('../data/metadata/DB3SubjectData.csv')
df_db3['Database'] = 'DB3'
# Deducir género
df_db3['Gender'] = df_db3.apply(
    lambda row: 'Female' if row['Height'] < 170 and row['Weight'] < 75 else 'Male', axis=1
)

In [35]:
# DB7 amputated (subjects 21-22)
df_db7_amp = df_db7[df_db7['Subject'] >= 21].copy()
# DB7 no tiene Phantom Limb Sensation y DASH Score
df_db7_amp['Phantom Limb Sensation Intensity'] = np.nan
df_db7_amp['DASH Score'] = np.nan

In [36]:
# Asegurar mismas columnas
cols_db3 = df_db3.columns.tolist()
for col in cols_db3:
    if col not in df_db7_amp.columns:
        df_db7_amp[col] = np.nan
df_db7_amp = df_db7_amp[cols_db3]

## 7. Guardar

In [37]:
intact_subjects = pd.concat(dfs_intact, ignore_index=True)
intact_subjects.to_csv('../data/master/intact_subjects.csv', index=False)

amputated_subjects = pd.concat([df_db3, df_db7_amp], ignore_index=True)
amputated_subjects.to_csv('../data/master/amputated_subjects.csv', index=False)

## 8. Verificación

In [38]:
print(f"Intact subjects: {len(intact_subjects)}")
print(f"Amputated subjects: {len(amputated_subjects)}")

Intact subjects: 107
Amputated subjects: 13
